# Shift-aware selective NIDS: reduced-scope proof of concept

**Question:** if an ML intrusion detector is trained on some networks and deployed on an unseen one, can we (a) keep detection decent and (b) know when its predictions should not be trusted?

**What this notebook does** (binary task: benign vs attack):
1. Streams the NetFlow-v2 datasets in chunks and samples each domain to a laptop-friendly size (about 50/50 benign/attack, with weights saved to restore natural proportions).
2. Runs sanity checks (duplicates, shortcut features, shuffled-label control).
3. **Leave-one-dataset-out**: each dataset takes a turn as the unseen network.
4. Compares classifiers and normalization modes, calibrates confidence three ways (none, standard, **leave-domain-out**), and measures selective risk.
5. Measures two shift alarms: a **flow-level** score (uncertainty, Mahalanobis, kNN) and a **window-level** MMD gate, including a synthetic-shift test with known severity.
6. Assigns ACCEPT / REVIEW / UNFAMILIAR_NETWORK and saves tables, figures and `sample_predictions.csv` for the Streamlit viewer.

**Not included (future work):** DANN / CORAL / GroupDRO, withheld attack families, multi-class training. This is a proof of concept, not a deployable system.

**Before running:** put the dataset file(s) (`.parquet` or `.csv`) in the folder set as `DATA_DIR` below.

In [ ]:
%pip install -q lightgbm pyarrow pandas numpy scikit-learn scipy matplotlib

In [ ]:
import glob, json, os, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import lightgbm as lgb
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")

# ------------------------------ CONFIG ------------------------------
DATA_DIR = "data"            # folder holding the uploaded dataset file(s)
OUT_DIR = "reports"          # tables, figures and sample_predictions.csv go here
N_PER_DOMAIN = 100_000       # rows kept per domain. Start small; raise to 300_000 once everything runs
BENIGN_FRAC = 0.5            # share of benign rows in each sampled domain
SEEDS = [0]                  # use [0, 1, 2] for the final run
# (classifier, normalization). "per_domain" uses the unlabeled test network's own statistics (test-time stats).
EXPERIMENTS = [("lgbm", "global"), ("lgbm", "per_domain"), ("lgbm_domainbal", "global")]
WINDOW = 500                 # flows per window for the network-level gate
N_WINDOWS = 60               # windows per group (threshold / control / held-out)
REF_SIZE = 2000              # reference flows for the MMD gate
SHIFT_LEVELS = [0.0, 0.05, 0.1, 0.2, 0.4]   # synthetic shift severity (std units per feature)
REBUILD_SAMPLE = False       # True forces re-sampling even if a cached sample exists
# Columns that identify hosts/networks or leak the label. Dropped (the dataset authors also dropped IDs, IPs, ports and TTLs).
DROP = {"ipv4_src_addr", "ipv4_dst_addr", "l4_src_port", "l4_dst_port", "min_ttl", "max_ttl",
        "dns_query_id", "flow_start_milliseconds", "flow_end_milliseconds"}
LABEL_NAMES = {"label", "attack", "dataset"}
PRIMARY = EXPERIMENTS[0]     # experiment used for sample_predictions.csv and the plots

os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)

## 1. Load and sample the data (streaming, so it fits in memory)
Pass 1 counts rows per (dataset, attack class). Pass 2 keeps a sample of each domain: about half benign, and the attack budget spread evenly over attack classes (rare classes are kept in full). Each row gets a weight = natural count / sampled count.

In [ ]:
CANON = {"unsw": "NF-UNSW-NB15-v2", "bot": "NF-BoT-IoT-v2", "ton": "NF-ToN-IoT-v2",
         "cse": "NF-CSE-CIC-IDS2018-v2", "2018": "NF-CSE-CIC-IDS2018-v2"}

def canon_domain(name):
    s = str(name).lower()
    for key, val in CANON.items():
        if key in s:
            return val
    return str(name)

def canon_series(ser):
    ser = ser.astype(str)
    mapping = {u: canon_domain(u) for u in ser.unique()}
    return ser.map(mapping)

def list_files():
    pats = ["**/*.parquet", "**/*.csv"]
    files = []
    for p in pats:
        files += glob.glob(os.path.join(DATA_DIR, p), recursive=True)
    return sorted(f for f in files if os.path.abspath(OUT_DIR) not in os.path.abspath(f))

def schema_cols(path):
    if path.lower().endswith(".parquet"):
        return list(pq.ParquetFile(path).schema_arrow.names)
    return [c.strip() for c in pd.read_csv(path, nrows=0).columns]

def find_col(cols, options):
    low = {c.lower(): c for c in cols}
    for o in options:
        if o in low:
            return low[o]
    return None

def iter_chunks(path, columns=None, batch=500_000):
    if path.lower().endswith(".parquet"):
        for b in pq.ParquetFile(path).iter_batches(batch_size=batch, columns=columns):
            yield b.to_pandas()
    else:
        for ch in pd.read_csv(path, chunksize=batch):
            ch.columns = [c.strip() for c in ch.columns]
            yield ch if columns is None else ch[columns]

def waterfill(counts, budget):
    """Split `budget` rows over classes as evenly as possible, never exceeding a class's size."""
    alloc = {k: 0 for k in counts}
    remaining = dict(counts)
    left = int(budget)
    while left > 0 and remaining:
        share = max(1, left // len(remaining))
        for k in list(remaining):
            take = min(share, remaining[k], left)
            alloc[k] += take
            remaining[k] -= take
            left -= take
            if remaining[k] == 0:
                del remaining[k]
            if left <= 0:
                break
    return alloc

def is_benign(a):
    return str(a).strip().lower() == "benign"

def build_sample():
    files = list_files()
    assert files, f"No .parquet/.csv files found under {DATA_DIR!r}. Upload the dataset first."
    print("Files:", files)
    info, counts = {}, {}
    t0 = time.time()
    for path in files:                                   # ---- pass 1: counts
        cols = schema_cols(path)
        dcol, acol, lcol = (find_col(cols, [n]) for n in ("dataset", "attack", "label"))
        assert acol and lcol, f"{path}: need 'Attack' and 'Label' columns, found {cols}"
        info[path] = (dcol, acol, lcol)
        need = [c for c in (dcol, acol) if c]
        for ch in iter_chunks(path, columns=need):
            dom = canon_series(ch[dcol]) if dcol else pd.Series(canon_domain(os.path.basename(path)), index=ch.index)
            att = ch[acol].astype(str).str.strip()
            g = pd.DataFrame({"d": dom.values, "a": att.values}).groupby(["d", "a"]).size()
            for (d, a), n in g.items():
                counts[(d, a)] = counts.get((d, a), 0) + int(n)
    domains = sorted({d for d, _ in counts})
    print(f"Pass 1 done in {time.time()-t0:.0f}s. Domains found: {domains}")
    keep_p, weight = {}, {}
    for d in domains:
        cls = {a: n for (dd, a), n in counts.items() if dd == d}
        ben = {a: n for a, n in cls.items() if is_benign(a)}
        att = {a: n for a, n in cls.items() if not is_benign(a)}
        alloc_b = waterfill(ben, int(N_PER_DOMAIN * BENIGN_FRAC))
        alloc_a = waterfill(att, N_PER_DOMAIN - sum(alloc_b.values()))
        for a, n in cls.items():
            take = {**alloc_b, **alloc_a}.get(a, 0)
            keep_p[f"{d}|{a}"] = take / n
            weight[f"{d}|{a}"] = n / max(take, 1)
    rng = np.random.default_rng(42)
    parts = []
    for path in files:                                   # ---- pass 2: sample
        dcol, acol, lcol = info[path]
        for ch in iter_chunks(path):
            dom = canon_series(ch[dcol]) if dcol else pd.Series(canon_domain(os.path.basename(path)), index=ch.index)
            att = ch[acol].astype(str).str.strip()
            key = dom + "|" + att
            p = key.map(keep_p).fillna(0.0).values
            mask = rng.random(len(ch)) < p
            if not mask.any():
                continue
            s = ch[mask].copy()
            s["_domain"] = dom[mask].values
            s["_attack"] = att[mask].values
            s["_w"] = key[mask].map(weight).values
            s["_y"] = pd.to_numeric(ch.loc[mask, lcol], errors="coerce").values
            parts.append(s)
    out = pd.concat(parts, ignore_index=True)
    print(f"Sampling done in {time.time()-t0:.0f}s: {len(out):,} rows kept")
    return out

CACHE = os.path.join(OUT_DIR, f"sampled_N{N_PER_DOMAIN}_b{int(BENIGN_FRAC*100)}.parquet")
if os.path.exists(CACHE) and not REBUILD_SAMPLE:
    raw = pd.read_parquet(CACHE)
    print("Loaded cached sample:", CACHE, f"({len(raw):,} rows)")
else:
    raw = build_sample()
    raw.to_parquet(CACHE, index=False)
    print("Saved sample to", CACHE)

## 2. Clean, harmonize and run sanity checks

In [ ]:
def family(a):
    """Coarse attack family (used only for reporting which attacks are missed). Document this mapping in any write-up."""
    s = str(a).strip().lower()
    if s == "benign": return "Benign"
    if "dos" in s: return "DoS_DDoS"
    if "recon" in s or "scan" in s: return "Recon_Scanning"
    if "brute" in s or "password" in s: return "BruteForce_Password"
    if "inject" in s or "xss" in s or "web" in s or "sql" in s: return "Web_Injection"
    if s in ("bot", "backdoor", "ransomware", "worms", "shellcode", "theft"): return "Malware_Botnet"
    return "Exploit_Other"

doms_present = sorted(raw["_domain"].unique())
common = set.intersection(*[set(raw.loc[raw["_domain"] == d].dropna(axis=1, how="all").columns) for d in doms_present])
feat_cols = sorted(c for c in common
                   if not c.startswith("_") and c.lower() not in DROP and c.lower() not in LABEL_NAMES
                   and pd.api.types.is_numeric_dtype(raw[c]))
print(f"{len(feat_cols)} feature columns kept:", feat_cols)

D, clean_log = {}, []
for d in doms_present:
    df = raw[raw["_domain"] == d]
    X = df[feat_cols].astype("float64").replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1) & df["_y"].isin([0, 1])
    X, df = X[ok], df[ok]
    out = X.astype("float32")
    out["y"] = df["_y"].astype(int).values
    out["w"] = df["_w"].astype(float).values
    out["attack"] = df["_attack"].values
    n_before = len(out)
    out = out.drop_duplicates(subset=feat_cols + ["y"]).reset_index(drop=True)
    out["family"] = out["attack"].map(family)
    D[d] = out
    clean_log.append({"domain": d, "rows_kept": len(out), "dropped_bad_rows": int((~ok).sum()),
                      "dropped_duplicates": n_before - len(out), "benign_share": 1 - out["y"].mean()})
DOMAINS = sorted(D)
display(pd.DataFrame(clean_log).round(3)) if "display" in globals() else print(pd.DataFrame(clean_log).round(3))
assert len(DOMAINS) >= 3, f"Need at least 3 domains for leave-one-out, found {DOMAINS}"

print("\nAttack classes per domain (sampled):")
for d in DOMAINS:
    print(f"  {d}: {D[d]['attack'].value_counts().to_dict()}")

In [ ]:
# --- Sanity check A: exact duplicate flows shared between domains (should be ~0) ---
hashes = {d: set(pd.util.hash_pandas_object(D[d][feat_cols], index=False).values) for d in DOMAINS}
print("Cross-domain duplicate flows:")
for i, a in enumerate(DOMAINS):
    for b in DOMAINS[i + 1:]:
        print(f"  {a} vs {b}: {len(hashes[a] & hashes[b])}")

# --- Sanity check B: can a model tell WHICH dataset a flow came from? (shortcut features) ---
rng0 = np.random.default_rng(0)
sub = pd.concat([D[d].sample(min(20000, len(D[d])), random_state=0).assign(_d=d) for d in DOMAINS])
codes, _ = pd.factorize(sub["_d"])
idx = rng0.permutation(len(sub)); cut = int(0.8 * len(idx))
dc = lgb.LGBMClassifier(n_estimators=100, num_leaves=31, verbose=-1, n_jobs=-1).fit(sub[feat_cols].values[idx[:cut]], codes[idx[:cut]])
acc = (dc.predict(sub[feat_cols].values[idx[cut:]]) == codes[idx[cut:]]).mean()
print(f"\nDomain-classifier accuracy: {acc:.3f}  (chance = {1/len(DOMAINS):.2f}). High = domains are easy to tell apart; the gate should detect shift easily.")
imp = pd.Series(dc.feature_importances_, index=feat_cols).sort_values(ascending=False).head(10)
print("Top features that identify the dataset (shortcut candidates):\n", imp.to_string())

## 3. Building blocks: normalization, models, calibration, metrics, MMD gate

In [ ]:
def log_signed(X):
    return np.sign(X) * np.log1p(np.abs(X))

XL = {d: log_signed(D[d][feat_cols].values.astype(np.float32)) for d in DOMAINS}
Y = {d: D[d]["y"].values.astype(int) for d in DOMAINS}
W = {d: D[d]["w"].values.astype(float) for d in DOMAINS}
FAM = {d: D[d]["family"].values for d in DOMAINS}

def normalize(XL, fit_idx, apply_domains, mode):
    """fit_idx: {domain: row indices used to fit statistics}. Only source-train rows may appear here.
    global     -> one mean/std from the pooled fit rows (source-only).
    per_domain -> each domain scaled by its OWN median/IQR. A domain without fit rows (the unseen network)
                  uses its own unlabeled data: this is 'test-time statistics' mode."""
    out = {}
    if mode == "global":
        pooled = np.vstack([XL[d][fit_idx[d]] for d in fit_idx])
        mu, sd = pooled.mean(0), pooled.std(0) + 1e-6
        for d in apply_domains:
            out[d] = np.clip((XL[d] - mu) / sd, -20, 20)
    elif mode == "per_domain":
        for d in apply_domains:
            ref = XL[d][fit_idx[d]] if d in fit_idx else XL[d]
            med = np.median(ref, 0)
            q75, q25 = np.percentile(ref, [75, 25], axis=0)
            iqr = q75 - q25
            sd = np.where(iqr > 1e-6, iqr, ref.std(0) + 1e-6)
            out[d] = np.clip((XL[d] - med) / sd, -20, 20)
    else:
        raise ValueError(mode)
    return out

def train_model(kind, X, y, dom_ids, seed):
    w = None
    if kind == "lgbm_domainbal":          # every source domain gets equal total weight
        cnt = pd.Series(dom_ids).value_counts()
        w = pd.Series(dom_ids).map(1.0 / cnt).values
        w = w / w.mean()
    m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=63, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, min_child_samples=50, reg_lambda=1.0, random_state=seed,
                           n_jobs=-1, verbose=-1)
    return m.fit(X, y, sample_weight=w)

def logit(m, X):
    return m.predict(X, raw_score=True)       # log-odds of "attack"

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

def fit_temperature(z, y):
    def nll(logT):
        p = np.clip(sigmoid(z / np.exp(logT)), 1e-7, 1 - 1e-7)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def ece(p, y, w=None, bins=15):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    w = np.ones_like(conf) if w is None else w
    edges = np.linspace(0.5, 1.0, bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
    tot, e = w.sum(), 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += (w[m].sum() / tot) * abs(np.average(correct[m], weights=w[m]) - np.average(conf[m], weights=w[m]))
    return float(e)

def reliability_curve(p, y, bins=10):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    idx = np.clip(np.digitize(conf, np.linspace(0.5, 1, bins + 1)[1:-1]), 0, bins - 1)
    xs, ys = [], []
    for b in range(bins):
        m = idx == b
        if m.sum() >= 20:
            xs.append(conf[m].mean()); ys.append(correct[m].mean())
    return xs, ys

def unc(z, T):
    p = sigmoid(z / T)
    return 1.0 - np.maximum(p, 1 - p)         # 0 = sure, 0.5 = coin flip

def risk_coverage(u, err):
    e = err[np.argsort(u, kind="stable")]
    risk = np.cumsum(e) / np.arange(1, len(e) + 1)
    cov = np.arange(1, len(e) + 1) / len(e)
    return cov, risk

_trapz = getattr(np, "trapezoid", None) or np.trapz

def aurc(cov, risk):
    return float(_trapz(risk, cov))

def sqdist(A, B):
    return cdist(A, B, "sqeuclidean")

class MMDGate:
    """Unbiased RBF-MMD^2 between a window of flows and a reference sample of source flows."""
    def __init__(self, R, rng):
        self.R = R
        sub = R[rng.choice(len(R), min(500, len(R)), replace=False)]
        d = sqdist(sub, sub)
        med = np.median(d[d > 0]) if (d > 0).any() else 1.0
        self.m = float(med)
        Krr = np.exp(-sqdist(R, R) / self.m)
        n = len(R)
        self.krr = (Krr.sum() - np.trace(Krr)) / (n * (n - 1))
    def score(self, Wd):
        m = len(Wd)
        Kww = np.exp(-sqdist(Wd, Wd) / self.m)
        Kwr = np.exp(-sqdist(Wd, self.R) / self.m)
        kww = (Kww.sum() - np.trace(Kww)) / (m * (m - 1))
        return float(kww + self.krr - 2 * Kwr.mean())

### Unit tests (these must pass before you trust any result)

In [ ]:
# ECE: 90% confident and 90% correct -> ~0 ; 99% confident but 50% correct -> large
p_ok = np.r_[np.full(900, 0.9), np.full(100, 0.1)]; y_ok = np.r_[np.ones(900), np.zeros(100)].astype(int)
assert ece(np.full(1000, 0.9), np.r_[np.ones(900), np.zeros(100)].astype(int)) < 0.01
assert ece(np.full(1000, 0.99), np.r_[np.ones(500), np.zeros(500)].astype(int)) > 0.4
# Temperature scaling recovers a known over-confidence factor
rt = np.random.default_rng(1); z_true = rt.normal(0, 2, 20000); y_t = (rt.random(20000) < sigmoid(z_true)).astype(int)
assert 2.5 < fit_temperature(3 * z_true, y_t) < 3.5
# waterfill never exceeds class size and uses the budget
a = waterfill({"x": 10, "y": 1000, "z": 1000}, 500); assert a["x"] == 10 and sum(a.values()) == 500
# MMD: same distribution ~0, shifted distribution clearly larger
rg = np.random.default_rng(2); Rr = rg.normal(size=(1000, 10)); gt = MMDGate(Rr, rg)
assert abs(gt.score(rg.normal(size=(300, 10)))) < 0.02 and gt.score(rg.normal(size=(300, 10)) + 0.5) > 0.05
# normalize(): the unseen domain must not influence global statistics
_fi = {d: np.arange(len(XL[d])) for d in DOMAINS[1:]}
_g1 = normalize(XL, _fi, DOMAINS, "global")[DOMAINS[1]]
_XL2 = dict(XL); _XL2[DOMAINS[0]] = XL[DOMAINS[0]] * 5 + 100
assert np.allclose(_g1, normalize(_XL2, _fi, DOMAINS, "global")[DOMAINS[1]]), "LEAK: held-out data changed the normalizer"
print("All unit tests passed.")

### Sanity check C: shuffled-label control
Training on shuffled labels must give AUROC near 0.5 on an unseen domain. If it does not, something is leaking.

In [ ]:
S0 = DOMAINS[1:]; H0 = DOMAINS[0]
fi0 = {d: np.arange(len(XL[d])) for d in S0}
X0 = normalize(XL, fi0, DOMAINS, "global")
yshuf = np.random.default_rng(0).permutation(np.concatenate([Y[d] for d in S0]))
m0 = lgb.LGBMClassifier(n_estimators=100, verbose=-1, n_jobs=-1).fit(np.vstack([X0[d] for d in S0]), yshuf)
auc0 = roc_auc_score(Y[H0], logit(m0, X0[H0]))
print(f"Shuffled-label AUROC on {H0}: {auc0:.3f}  (expect about 0.5)")
assert 0.4 < auc0 < 0.6, "Shuffled-label control failed: investigate leakage"

## 4. The experiment: leave-one-dataset-out
For each held-out domain H: train on the other domains (80% train / 20% validation). Calibrate three ways. Run the shift alarms. Everything that picks a threshold uses source data only.

In [ ]:
def make_splits(S, seed):
    rng = np.random.default_rng(1000 + seed)
    sp = {}
    for s in S:
        perm = rng.permutation(len(XL[s])); cut = int(0.8 * len(perm))
        sp[s] = {"train": np.sort(perm[:cut]), "val": np.sort(perm[cut:])}
    return sp

def shift_detection(H, S, sp, XG, XP, seed):
    """Window-level MMD gate (global space) and flow-level OOD scores in two feature spaces."""
    rng = np.random.default_rng(2000 + seed)
    val_dom = np.concatenate([np.repeat(s, len(sp[s]["val"])) for s in S])
    XG_val = np.vstack([XG[s][sp[s]["val"]] for s in S])
    tr = np.vstack([XG[s][sp[s]["train"]] for s in S])
    R = tr[rng.choice(len(tr), min(REF_SIZE, len(tr)), replace=False)]
    gate = MMDGate(R, rng)
    perm = rng.permutation(len(XG_val)); half = len(perm) // 2
    A, B = perm[:half], perm[half:]
    assert len(A) >= WINDOW and len(XG[H]) >= WINDOW, "Not enough rows for the chosen WINDOW size"
    winA = [rng.choice(A, WINDOW, replace=False) for _ in range(N_WINDOWS)]
    winB = [rng.choice(B, WINDOW, replace=False) for _ in range(N_WINDOWS)]
    winH = [rng.choice(len(XG[H]), WINDOW, replace=False) for _ in range(N_WINDOWS)]
    sA = np.array([gate.score(XG_val[w]) for w in winA])
    tau = float(np.percentile(sA, 95))                      # threshold from SOURCE windows only
    sB = np.array([gate.score(XG_val[w]) for w in winB])    # control: familiar windows never used for the threshold
    sH = np.array([gate.score(XG[H][w]) for w in winH])
    v = rng.normal(size=XG_val.shape[1])
    syn = {s: float(np.mean([gate.score(XG_val[w] + s * v) > tau for w in winB])) for s in SHIFT_LEVELS}
    ood = {}
    for space, X in (("global", XG), ("per_domain", XP)):
        trs = np.vstack([X[s][sp[s]["train"]] for s in S]); vas = np.vstack([X[s][sp[s]["val"]] for s in S])
        neg = vas[rng.choice(len(vas), min(5000, len(vas)), replace=False)]
        pos = X[H][rng.choice(len(X[H]), min(5000, len(X[H])), replace=False)]
        mu = trs.mean(0); P = np.linalg.pinv(np.cov(trs, rowvar=False) + 1e-3 * np.eye(trs.shape[1]))
        maha = lambda Z: (((Z - mu) @ P) * (Z - mu)).sum(1)
        ref = trs[rng.choice(len(trs), min(20000, len(trs)), replace=False)]
        nn = NearestNeighbors(n_neighbors=10, algorithm="brute", n_jobs=-1).fit(ref)
        knn = lambda Z: nn.kneighbors(Z)[0].mean(1)
        lab = np.r_[np.zeros(len(neg)), np.ones(len(pos))]
        ood[f"auroc_mahalanobis_{space}"] = roc_auc_score(lab, np.r_[maha(neg), maha(pos)])
        ood[f"auroc_knn_{space}"] = roc_auc_score(lab, np.r_[knn(neg), knn(pos)])
    return dict(tau=tau, sA=sA, sB=sB, sH=sH, winB=winB, winH=winH, syn=syn, ood=ood,
                val_dom=val_dom, gate_auroc=roc_auc_score(np.r_[np.zeros(len(sB)), np.ones(len(sH))], np.r_[sB, sH]))

def run_experiment(kind, norm, H, S, sp, shift, seed):
    fit_idx = {s: sp[s]["train"] for s in S}
    assert H not in fit_idx, "LEAK: held-out domain in fit set"
    X = normalize(XL, fit_idx, S + [H], norm)
    Xtr = np.vstack([X[s][sp[s]["train"]] for s in S])
    ytr = np.concatenate([Y[s][sp[s]["train"]] for s in S])
    dtr = np.concatenate([np.repeat(s, len(sp[s]["train"])) for s in S])
    model = train_model(kind, Xtr, ytr, dtr, seed)
    z_val = np.concatenate([logit(model, X[s][sp[s]["val"]]) for s in S])
    y_val = np.concatenate([Y[s][sp[s]["val"]] for s in S])
    zH, yH, wH = logit(model, X[H]), Y[H], W[H]
    T_std = fit_temperature(z_val, y_val)                     # standard: in-distribution validation
    zl, yl = [], []                                           # leave-domain-out: each source plays the unseen network
    for k in S:
        rest = [s for s in S if s != k]
        fi = {s: np.arange(len(XL[s])) for s in rest}
        Xin = normalize(XL, fi, rest + [k], norm)
        mk = train_model(kind, np.vstack([Xin[s] for s in rest]), np.concatenate([Y[s] for s in rest]),
                         np.concatenate([np.repeat(s, len(Y[s])) for s in rest]), seed)
        zl.append(logit(mk, Xin[k])); yl.append(Y[k])
    T_ldo = fit_temperature(np.concatenate(zl), np.concatenate(yl))

    pred = (zH >= 0).astype(int)
    base = dict(heldout=H, seed=seed, model=kind, norm=norm,
                auroc=roc_auc_score(yH, zH), macro_f1=f1_score(yH, pred, average="macro"),
                tpr=pred[yH == 1].mean(), fpr=pred[yH == 0].mean(), T_std=T_std, T_ldo=T_ldo,
                indomain_auroc=roc_auc_score(y_val, z_val),
                indomain_f1=f1_score(y_val, (z_val >= 0).astype(int), average="macro"),
                indomain_ece_uncal=ece(sigmoid(z_val), y_val))
    for name, T in (("uncal", 1.0), ("std", T_std), ("ldo", T_ldo)):
        p = sigmoid(zH / T)
        base[f"ece_{name}"] = ece(p, yH); base[f"ecew_{name}"] = ece(p, yH, wH)
        base[f"brier_{name}"] = brier_score_loss(yH, p)
    rng = np.random.default_rng(3000 + seed)
    u_val, u_H = unc(z_val, T_ldo), unc(zH, T_ldo)
    neg = rng.choice(len(u_val), min(5000, len(u_val)), replace=False); pos = rng.choice(len(u_H), min(5000, len(u_H)), replace=False)
    base["auroc_uncertainty_ood"] = roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[u_val[neg], u_H[pos]])
    err = (pred != yH).astype(float)
    cov, risk = risk_coverage(u_H, err)
    base["aurc"] = aurc(cov, risk); base["risk_cov100"] = float(risk[-1])
    for c in (0.8, 0.9):
        base[f"risk_at_{int(c*100)}"] = float(risk[int(c * len(risk)) - 1])
    tau_u = float(np.percentile(u_val, 90))                   # REVIEW threshold: source validation only
    base["err_if_confident"] = float(err[u_H <= tau_u].mean()) if (u_H <= tau_u).any() else np.nan
    base["err_if_uncertain"] = float(err[u_H > tau_u].mean()) if (u_H > tau_u).any() else np.nan
    for fam in sorted(set(FAM[H]) - {"Benign"}):               # which attack families are missed on the unseen network
        base[f"tpr_{fam}"] = float(pred[(FAM[H] == fam)].mean())

    # ---- decision layer: window gate first, then flow uncertainty ----
    def decide(fire, u):
        return np.where(fire, "UNFAMILIAR_NETWORK", np.where(u > tau_u, "REVIEW", "ACCEPT"))
    idxH = np.concatenate(shift["winH"]); idxB = np.concatenate(shift["winB"])
    stH = decide(np.repeat(shift["sH"] > shift["tau"], WINDOW), u_H[idxH])
    stB = decide(np.repeat(shift["sB"] > shift["tau"], WINDOW), u_val[idxB])
    cross = []
    for origin, st in (("unseen network", stH), ("familiar control", stB)):
        row = dict(origin=origin, heldout=H, seed=seed, model=kind, norm=norm)
        for s in ("ACCEPT", "REVIEW", "UNFAMILIAR_NETWORK"):
            row[s] = float(np.mean(st == s))
        cross.append(row)
    sample = None
    if (kind, norm) == PRIMARY:
        def mk(dom, st, z, y, ws, origin):
            p = sigmoid(z / T_ldo)
            return pd.DataFrame(dict(domain=dom, heldout_fold=H, origin=origin, status=st,
                                     prediction=np.where(p >= 0.5, "Attack", "Normal"),
                                     confidence=np.maximum(p, 1 - p).round(4),
                                     true_label=np.where(y == 1, "Attack", "Normal"), window_score=ws))
        a = mk(np.repeat(f"{H} (unseen)", len(idxH)), stH, zH[idxH], yH[idxH], np.repeat(shift["sH"], WINDOW), "unseen")
        b = mk(shift["val_dom"][idxB] + " (familiar)", stB, z_val[idxB], y_val[idxB], np.repeat(shift["sB"], WINDOW), "familiar")
        sample = pd.concat([a.sample(min(2000, len(a)), random_state=seed), b.sample(min(2000, len(b)), random_state=seed)])
    plot = dict(rel={n: reliability_curve(sigmoid(zH / T), yH) for n, T in (("uncalibrated", 1.0), ("standard TS", T_std), ("leave-domain-out TS", T_ldo))},
                cov=cov, risk=risk)
    return base, cross, sample, plot

In [ ]:
RESULTS, CROSS, SAMPLES, SHIFTS, PLOTS = [], [], [], [], {}
t_start = time.time()
for seed in SEEDS:
    for H in DOMAINS:
        S = [d for d in DOMAINS if d != H]
        sp = make_splits(S, seed)
        fit_idx = {s: sp[s]["train"] for s in S}
        XG = normalize(XL, fit_idx, DOMAINS, "global")         # held-out domain scaled with SOURCE statistics
        XP = normalize(XL, fit_idx, DOMAINS, "per_domain")     # held-out domain scaled with its own unlabeled data
        shift = shift_detection(H, S, sp, XG, XP, seed)
        SHIFTS.append(dict(heldout=H, seed=seed, tau=shift["tau"], gate_auroc=shift["gate_auroc"],
                           detect_heldout=float(np.mean(shift["sH"] > shift["tau"])),
                           false_alarm_control=float(np.mean(shift["sB"] > shift["tau"])),
                           **{f"syn_{s}": v for s, v in shift["syn"].items()}, **shift["ood"]))
        PLOTS[(H, seed, "shift")] = shift
        for kind, norm in EXPERIMENTS:
            base, cross, sample, plot = run_experiment(kind, norm, H, S, sp, shift, seed)
            RESULTS.append(base); CROSS += cross
            if sample is not None: SAMPLES.append(sample)
            if (kind, norm) == PRIMARY: PLOTS[(H, seed, "plot")] = plot
            print(f"[{time.time()-t_start:6.0f}s] seed={seed} heldout={H:24s} {kind:15s} {norm:10s} "
                  f"AUROC={base['auroc']:.3f} ECE {base['ece_uncal']:.3f}->{base['ece_ldo']:.3f}")
print("Done.")

## 5. Results

In [ ]:
res = pd.DataFrame(RESULTS); shifts = pd.DataFrame(SHIFTS); cross = pd.DataFrame(CROSS)
res.to_csv(os.path.join(OUT_DIR, "results_by_fold.csv"), index=False)
shifts.to_csv(os.path.join(OUT_DIR, "shift_detection.csv"), index=False)
cross.to_csv(os.path.join(OUT_DIR, "status_crosstab.csv"), index=False)
if SAMPLES:
    pd.concat(SAMPLES).to_csv(os.path.join(OUT_DIR, "sample_predictions.csv"), index=False)

def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

print("1) Classification on the unseen network vs in-domain validation (means over folds/seeds)")
show(res.groupby(["model", "norm"])[["indomain_auroc", "auroc", "indomain_f1", "macro_f1", "tpr", "fpr"]].mean().round(3))
print("\n2) Calibration on the unseen network (lower is better). ECE-w restores natural class proportions.")
show(res.groupby(["model", "norm"])[["ece_uncal", "ece_std", "ece_ldo", "ecew_uncal", "ecew_std", "ecew_ldo", "T_std", "T_ldo"]].mean().round(3))
print("\n3) Selective prediction on the unseen network (lower risk/AURC is better)")
show(res.groupby(["model", "norm"])[["risk_cov100", "risk_at_90", "risk_at_80", "aurc", "err_if_confident", "err_if_uncertain", "auroc_uncertainty_ood"]].mean().round(3))
print("\n4) Per-fold detail (primary experiment)")
show(res[(res.model == PRIMARY[0]) & (res.norm == PRIMARY[1])][["heldout", "seed", "auroc", "macro_f1", "tpr", "fpr", "ece_uncal", "ece_std", "ece_ldo"]].round(3))
print("\n5) Shift alarms: window gate and flow-level OOD AUROC (global vs per-domain-normalized space)")
show(shifts[["heldout", "gate_auroc", "detect_heldout", "false_alarm_control", "auroc_mahalanobis_global", "auroc_mahalanobis_per_domain", "auroc_knn_global", "auroc_knn_per_domain"]].round(3))
print("\n6) Synthetic shift: fraction of familiar windows flagged as severity grows (should rise; severity 0 = false-alarm rate)")
show(shifts[["heldout"] + [f"syn_{s}" for s in SHIFT_LEVELS]].round(3))
print("\n7) Final status mix by data origin (unseen windows should be flagged; familiar ones mostly ACCEPT)")
show(cross.groupby(["model", "norm", "origin"])[["ACCEPT", "REVIEW", "UNFAMILIAR_NETWORK"]].mean().round(3))
fam_cols = [c for c in res.columns if c.startswith("tpr_")]
if fam_cols:
    print("\n8) Detection rate by attack family on the unseen network (primary experiment)")
    show(res[(res.model == PRIMARY[0]) & (res.norm == PRIMARY[1])].groupby("heldout")[fam_cols].mean().round(3))

### Figures

In [ ]:
seed0 = SEEDS[0]
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4.2 * len(DOMAINS), 4), sharey=True)
for ax, H in zip(np.atleast_1d(axes), DOMAINS):
    for name, (xs, ys) in PLOTS[(H, seed0, "plot")]["rel"].items():
        ax.plot(xs, ys, marker="o", label=name)
    ax.plot([0.5, 1], [0.5, 1], "k--", lw=1); ax.set_title(f"unseen: {H}", fontsize=9); ax.set_xlabel("stated confidence")
np.atleast_1d(axes)[0].set_ylabel("actual accuracy"); np.atleast_1d(axes)[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "reliability_diagrams.png"), dpi=130); plt.show()

fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4.2 * len(DOMAINS), 3.6), sharey=True)
for ax, H in zip(np.atleast_1d(axes), DOMAINS):
    pl = PLOTS[(H, seed0, "plot")]; ax.plot(pl["cov"], pl["risk"]); ax.set_title(f"unseen: {H}", fontsize=9); ax.set_xlabel("coverage (share of flows answered)")
np.atleast_1d(axes)[0].set_ylabel("error rate on answered flows")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "risk_coverage.png"), dpi=130); plt.show()

fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4.2 * len(DOMAINS), 3.6))
for ax, H in zip(np.atleast_1d(axes), DOMAINS):
    sh = PLOTS[(H, seed0, "shift")]
    ax.hist(sh["sB"], bins=15, alpha=.6, label="familiar windows"); ax.hist(sh["sH"], bins=15, alpha=.6, label="unseen windows")
    ax.axvline(sh["tau"], color="k", ls="--", lw=1); ax.set_title(f"unseen: {H}", fontsize=9); ax.set_xlabel("window MMD$^2$")
np.atleast_1d(axes)[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "window_scores.png"), dpi=130); plt.show()

plt.figure(figsize=(5, 3.6))
for H in DOMAINS:
    sy = PLOTS[(H, seed0, "shift")]["syn"]; plt.plot(list(sy), list(sy.values()), marker="o", label=H)
plt.xlabel("synthetic shift severity"); plt.ylabel("fraction of windows flagged"); plt.legend(fontsize=6); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figures", "synthetic_shift.png"), dpi=130); plt.show()

## 6. Automatic sanity verdicts
These are heuristic checks (my rules of thumb, not published standards). A `CHECK` means look closer, not that the result is wrong.

In [ ]:
prim = res[(res.model == PRIMARY[0]) & (res.norm == PRIMARY[1])]
gap = (prim.indomain_auroc - prim.auroc).mean()
print(("PASS " if gap > 0.01 else "CHECK"), f"in-domain vs unseen AUROC gap = {gap:.3f} (no gap at all would suggest leakage)")
print(("PASS " if (prim.auroc < 0.999).all() else "CHECK"), f"max unseen AUROC = {prim.auroc.max():.4f} (near-perfect scores are suspicious)")
better = (prim.ece_ldo < prim.ece_std).mean()
print(("PASS " if better >= 0.5 else "CHECK"), f"leave-domain-out calibration beats standard calibration in {better:.0%} of folds")
print(("PASS " if shifts.detect_heldout.mean() > 0.8 else "CHECK"), f"window gate flags {shifts.detect_heldout.mean():.0%} of unseen windows")
print(("PASS " if shifts.false_alarm_control.mean() < 0.2 else "CHECK"), f"window gate false-alarm rate on familiar windows = {shifts.false_alarm_control.mean():.0%}")
syn_cols = [f"syn_{s}" for s in SHIFT_LEVELS]
mono = (shifts[syn_cols].diff(axis=1).iloc[:, 1:] >= -0.1).all().all()
print(("PASS " if mono else "CHECK"), "synthetic-shift detection rises with severity")
print(("PASS " if (prim.err_if_uncertain > prim.err_if_confident).mean() >= 0.5 else "CHECK"), "errors are more common among flows marked uncertain than confident")
print("\nReminder: these four datasets differ hugely, so detecting 'unseen network' is expected to be easy. The synthetic-shift table is the harder test.")

## 7. Files written
Everything is saved to the `reports/` folder. To browse `sample_predictions.csv`, tables and figures in a web page, run the Streamlit viewer: `python -m streamlit run app.py`.

In [ ]:
with open(os.path.join(OUT_DIR, "run_config.json"), "w") as f:
    json.dump(dict(N_PER_DOMAIN=N_PER_DOMAIN, BENIGN_FRAC=BENIGN_FRAC, SEEDS=SEEDS, EXPERIMENTS=EXPERIMENTS, WINDOW=WINDOW,
                   N_WINDOWS=N_WINDOWS, REF_SIZE=REF_SIZE, domains=DOMAINS, features=feat_cols), f, indent=2)
for p in sorted(glob.glob(os.path.join(OUT_DIR, "**", "*.*"), recursive=True)):
    print(p)